# Market Basket - matrice fattura-prodotto (United Kingdom)

Notebook preparatorio per l'analisi di market basket su Online Retail II: costruisce la matrice binaria fattura x prodotto, limitata al Regno Unito per contenere le dimensioni, usando una rappresentazione **sparsa** per contenere l'uso di memoria.

Questo notebook si ferma alla costruzione e verifica della matrice: l'estrazione delle regole di associazione (es. algoritmo Apriori) e' un passo successivo, non incluso qui.

In [1]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder

## 1. Caricamento dati puliti

In [2]:
CLEAN_PATH = "../../data/processed/online_retail_clean.csv"

df = pd.read_csv(CLEAN_PATH)
print(f"Righe totali: {len(df):,}")
df.head()

Righe totali: 779,423


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country,TotalPrice
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.0


## 2. Filtro United Kingdom

Il Regno Unito rappresenta la quota largamente maggioritaria delle transazioni. Limitare l'analisi a questo mercato riduce sensibilmente il numero di fatture e di codici prodotto unici, mantenendo comunque un campione ampio e rappresentativo.

In [3]:
uk = df[df["Country"] == "United Kingdom"].copy()

print(f"Righe UK: {len(uk):,} su {len(df):,} totali ({len(uk) / len(df) * 100:.1f}%)")
print(f"Fatture uniche (Invoice): {uk['Invoice'].nunique():,}")
print(f"Codici prodotto unici (StockCode): {uk['StockCode'].nunique():,}")

Righe UK: 700,386 su 779,423 totali (89.9%)


Fatture uniche (Invoice): 33,539
Codici prodotto unici (StockCode): 4,615


## 3. Costruzione della matrice fattura-prodotto (sparsa)

Per ogni fattura costruiamo la lista dei prodotti distinti acquistati (una "transazione"), poi usiamo `TransactionEncoder` di **mlxtend** con `sparse=True`: invece di generare un array denso (una cella per ogni combinazione fattura x prodotto, con quasi tutte False), restituisce direttamente una matrice sparsa `scipy.sparse.csr_matrix`, che memorizza solo le combinazioni effettivamente presenti.

La differenza e' sostanziale: con circa 33.000 fatture e 4.600 codici prodotto, una matrice densa richiederebbe centinaia di MB di memoria, a fronte di una densita' reale attorno allo 0.4-0.5% (la maggior parte delle combinazioni fattura-prodotto non esiste).

In [4]:
transactions_by_invoice = uk.groupby("Invoice")["StockCode"].apply(lambda s: list(s.unique()))
transactions = transactions_by_invoice.tolist()
invoice_ids = transactions_by_invoice.index

print(f"Numero di transazioni (fatture): {len(transactions):,}")
print(f"Esempio transazione: {transactions[0][:5]}")

Numero di transazioni (fatture): 33,539
Esempio transazione: ['85048', '79323P', '79323W', '22041', '21232']


In [5]:
te = TransactionEncoder()
sparse_matrix = te.fit_transform(transactions, sparse=True)

basket = pd.DataFrame.sparse.from_spmatrix(
    sparse_matrix, index=invoice_ids, columns=te.columns_
)

basket.head()

,10002,10080,10109,10120,10123C,10123G,10124A,10124G,10125,10133,...,BANK CHARGES,C2,D,DOT,M,PADS,POST,SP1002,TEST001,TEST002
Invoice,,,,,,,,,,,,,,,,,,,,,
489434,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
489435,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
489436,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
489437,True,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
489438,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


## 4. Dimensioni della matrice risultante

In [6]:
n_invoices, n_products = basket.shape
n_present = sparse_matrix.nnz
density = n_present / (n_invoices * n_products)

dense_bool_mb = (n_invoices * n_products) / 1_000_000  # 1 byte per cella in un array bool denso
sparse_mb = basket.memory_usage(deep=True).sum() / 1_000_000

print(f"Dimensioni matrice: {n_invoices:,} fatture x {n_products:,} prodotti")
print(f"Combinazioni fattura-prodotto presenti: {n_present:,}")
print(f"Densita': {density * 100:.3f}%")
print()
print(f"Memoria stimata come array booleano denso: ~{dense_bool_mb:.1f} MB")
print(f"Memoria effettiva rappresentazione sparsa:  ~{sparse_mb:.1f} MB")

Dimensioni matrice: 33,539 fatture x 4,615 prodotti
Combinazioni fattura-prodotto presenti: 689,971
Densita': 0.446%

Memoria stimata come array booleano denso: ~154.8 MB
Memoria effettiva rappresentazione sparsa:  ~3.7 MB


## 5. Itemset frequenti con Apriori

Applichiamo `apriori` direttamente sulla matrice sparsa `basket`, partendo da `min_support=0.02` (un itemset deve comparire in almeno il 2% delle fatture per essere considerato "frequente").

Questo valore si e' rivelato un buon punto di partenza: il calcolo richiede pochi secondi e produce un numero di itemset sufficiente a generare piu' regole di quante ne servano per la selezione finale, senza scendere a soglie cosi' basse da includere coppie di prodotti co-acquistate per puro caso.

In [7]:
from mlxtend.frequent_patterns import apriori, association_rules

frequent_itemsets = apriori(basket, min_support=0.02, use_colnames=True)

print(f"Itemset frequenti trovati: {len(frequent_itemsets):,}")
print(frequent_itemsets["itemsets"].apply(len).value_counts().sort_index().rename("n_itemset"))

Itemset frequenti trovati: 221
itemsets
1    193
2     28
Name: n_itemset, dtype: int64


## 6. Regole di associazione

Generiamo tutte le regole possibili dagli itemset frequenti (senza soglia minima su confidence, per non scartare nulla in questa fase), calcolando `support`, `confidence` e `lift`. Filtriamo poi solo le regole con **lift > 1**: un lift maggiore di 1 indica che i due prodotti vengono acquistati insieme piu' spesso di quanto ci si aspetterebbe se fossero indipendenti - le uniche regole realmente informative per suggerimenti di cross-selling.

In [8]:
rules = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=0.0,
    num_itemsets=len(frequent_itemsets),
)

rules = rules[rules["lift"] > 1].sort_values("lift", ascending=False).reset_index(drop=True)

print(f"Regole generate: {len(rules):,}")

Regole generate: 56


## 7. Le 10 regole piu' interessanti (con nomi prodotto)

Le regole di Apriori lavorano sugli `StockCode`, poco leggibili di per se'. Costruiamo una mappa `StockCode -> Description` (usando la descrizione piu' frequente per codice, dato che in un dataset reale lo stesso codice puo' avere piccole varianti testuali nella descrizione) e la usiamo per tradurre antecedenti e conseguenti in nomi prodotto.

In [9]:
description_map = uk.groupby("StockCode")["Description"].agg(lambda s: s.value_counts().idxmax())


def to_names(itemset):
    return ", ".join(sorted(description_map.get(code, code) for code in itemset))


rules["Prodotti (antecedenti)"] = rules["antecedents"].apply(to_names)
rules["Prodotti (conseguenti)"] = rules["consequents"].apply(to_names)

In [10]:
top10 = rules.head(10)[
    ["Prodotti (antecedenti)", "Prodotti (conseguenti)", "support", "confidence", "lift"]
].round(3)

top10

,Prodotti (antecedenti),Prodotti (conseguenti),support,confidence,lift
0,SWEETHEART CERAMIC TRINKET BOX,STRAWBERRY CERAMIC TRINKET BOX,0.024,0.722,14.094
1,STRAWBERRY CERAMIC TRINKET BOX,SWEETHEART CERAMIC TRINKET BOX,0.024,0.461,14.094
2,WOODEN FRAME ANTIQUE WHITE,WOODEN PICTURE FRAME WHITE FINISH,0.029,0.565,11.772
3,WOODEN PICTURE FRAME WHITE FINISH,WOODEN FRAME ANTIQUE WHITE,0.029,0.603,11.772
4,LOVE BUILDING BLOCK WORD,HOME BUILDING BLOCK WORD,0.023,0.528,10.023
5,HOME BUILDING BLOCK WORD,LOVE BUILDING BLOCK WORD,0.023,0.432,10.023
6,HEART OF WICKER LARGE,HEART OF WICKER SMALL,0.025,0.497,9.631
7,HEART OF WICKER SMALL,HEART OF WICKER LARGE,0.025,0.494,9.631
8,LUNCH BAG CARS BLUE,LUNCH BAG PINK POLKADOT,0.022,0.455,8.994
9,LUNCH BAG PINK POLKADOT,LUNCH BAG CARS BLUE,0.022,0.438,8.994


## 8. Interpretazione business

**1. Portagioie in ceramica "Strawberry" <-> "Sweetheart" (lift ~14.1, il piu' alto)** - sono due varianti dello stesso oggetto decorativo, stessa forma e funzione ma fantasia diversa. Il lift altissimo suggerisce che chi acquista un portagioie di questa linea tende a comprarne piu' di uno per completare/regalare la collezione. E' un caso da manuale di cross-sell: mostrare in pagina prodotto le altre varianti della stessa linea.

**2. Cuori decorativi in vimini, taglia Large <-> Small (lift ~9.6)** - coppia dimensionale dello stesso articolo. L'associazione forte indica che il cliente non sceglie "una taglia", ma acquista il set completo (probabilmente per comporre una decorazione a piu' elementi). Qui ha senso vendere le due taglie come bundle a prezzo scontato invece che come due articoli separati.

**3. Lunch bag "Cars Blue" <-> "Pink Polkadot" (lift ~9.0)** - a differenza dei casi precedenti, questi non sono la stessa fantasia in varianti diverse, ma due design con target estetico opposto (tipicamente associato a bambino/bambina). L'associazione suggerisce acquisti per piu' figli nello stesso ordine, non una preferenza di gusto personale: un segnale utile per capire che il cliente tipo di questo prodotto e' spesso un genitore con piu' di un figlio, informazione rilevante per targeting e bundle famiglia.

In generale, le regole con lift piu' alto in questo dataset emergono quasi tutte da coppie "stesso prodotto, variante diversa" (fantasia o taglia): un pattern atteso per un retailer di articoli da regalo/decorazione, e un forte argomento a favore di raggruppare le varianti nella stessa scheda prodotto e proporle come bundle, piuttosto che trattarle come articoli scollegati nel catalogo.

## 9. Salvataggio delle regole di associazione

Salviamo tutte le regole con `lift > 1` (gia' filtrate e ordinate per lift decrescente nella sezione 6), con i nomi prodotto al posto degli StockCode, in `data/processed/market_basket_rules.csv`.

In [11]:
output_path = "../../data/processed/market_basket_rules.csv"

rules_export = rules[
    ["Prodotti (antecedenti)", "Prodotti (conseguenti)", "support", "confidence", "lift"]
].round(4)

rules_export.to_csv(output_path, index=False, encoding="utf-8")
print(f"Salvato in: {output_path}")
print(f"Regole salvate: {len(rules_export):,}")

Salvato in: ../../data/processed/market_basket_rules.csv
Regole salvate: 56
